# Kaggle version 2 — downloaded code with actual result tables
The five evaluation tables below are embedded from the verified CSV outputs of the completed Kaggle run.
These are actual results, not a local rerun. Original per-cell execution outputs are not provided by the pull API.
Source: https://www.kaggle.com/code/callingzeng/recsys-baby-full-20261113 (version 2)


# Kaggle version
Attach the original Baby dataset under `/kaggle/input`. Enable a GPU in Kaggle settings.
Run all cells for the full experiment. To run only pre-ranking, run setup and section 2; attach the complete original `recall/*.parquet` set as input.
Experiment definitions and training settings are unchanged. Outputs are under `/kaggle/working/recsys`.
Memory changes: chunk history accumulation and cross features; save/load three-tower models sequentially; release intermediate tables. No API calls are made by setup.


In [ ]:
from pathlib import Path
import shutil
import gc
import pickle
import tempfile
from collections.abc import MutableMapping

class DiskResults(MutableMapping):
    # Keep the dictionary interface and values, but load one experiment at a time.
    def __init__(self):
        self.directory = tempfile.TemporaryDirectory(prefix='recsys-')
        self.paths = {}
    def __getitem__(self, key):
        with self.paths[key].open('rb') as stream:
            return pickle.load(stream)
    def __setitem__(self, key, value):
        path = Path(self.directory.name) / (str(key) + '.pickle')
        with path.open('wb') as stream:
            pickle.dump(value, stream, protocol=pickle.HIGHEST_PROTOCOL)
        self.paths[key] = path
    def __delitem__(self, key):
        self.paths.pop(key).unlink()
    def __iter__(self):
        return iter(self.paths)
    def __len__(self):
        return len(self.paths)

# Set this explicitly only if multiple attached datasets contain baby.inter.
DATASET_DIR = None
WORK = Path('/kaggle/working/recsys')
RESULTS_ROOT = WORK / 'results'
RECALL_ROOT = WORK / 'recall'
for directory in (WORK, RESULTS_ROOT, RECALL_ROOT):
    directory.mkdir(parents=True, exist_ok=True)
if DATASET_DIR is None:
    matches = sorted(Path('/kaggle/input').rglob('baby.inter'))
    if len(matches) != 1:
        raise RuntimeError(f'Expected one baby.inter; found {matches}. Set DATASET_DIR explicitly.')
    INPUT_DATA_DIR = matches[0].parent
else:
    INPUT_DATA_DIR = Path(DATASET_DIR)
required = ['baby.inter', 'u_id_mapping.csv', 'i_id_mapping.csv',
            'image_feat.npy', 'text_feat.npy', 'user_graph_dict.npy']
missing = [name for name in required if not (INPUT_DATA_DIR / name).is_file()]
assert not missing, f'Missing input files: {missing}'
# Optional existing recall artifacts may be attached as a separate dataset.
recall_inputs = sorted(p for p in Path('/kaggle/input').rglob('*.parquet')
                       if p.parent.name == 'recall')
sources = {}
for source in recall_inputs:
    if source.name in sources:
        raise RuntimeError(f'Duplicate recall filename: {source.name}')
    sources[source.name] = source
    shutil.copy2(source, RECALL_ROOT / source.name)
print('Input:', INPUT_DATA_DIR, '| Output:', WORK)
print('Imported recall files:', len(sources))


# 0. Data Loading

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd


data_dir = INPUT_DATA_DIR

# 读取数据
inter = pd.read_csv(data_dir / "baby.inter", sep="\t")
user_map = pd.read_csv(data_dir / "u_id_mapping.csv", sep="\t")
item_map = pd.read_csv(data_dir / "i_id_mapping.csv", sep="\t")
image_feat = np.load(data_dir / "image_feat.npy", mmap_mode="r")
text_feat = np.load(data_dir / "text_feat.npy", mmap_mode="r")
user_graph = np.load(data_dir / "user_graph_dict.npy", allow_pickle=True)
user_graph = user_graph.item() if user_graph.shape == () else user_graph

# 数据集概览
summary = pd.DataFrame([
    ["baby.inter", str(inter.shape), "用户-商品交互", "用户ID、商品ID、评分、时间戳、x_label", "训练与评估推荐模型"],
    ["u_id_mapping.csv", str(user_map.shape), "用户ID映射", "Amazon原始用户ID → 数字ID", "用户编号转换"],
    ["i_id_mapping.csv", str(item_map.shape), "商品ID映射", "Amazon ASIN → 数字ID", "商品编号转换"],
    ["image_feat.npy", str(image_feat.shape), "商品图片特征", "每件商品的4096维视觉向量", "多模态推荐、视觉相似度"],
    ["text_feat.npy", str(text_feat.shape), "商品文本特征", "每件商品的384维文本向量", "多模态推荐、语义相似度"],
    ["user_graph_dict.npy", f"{len(user_graph)} keys" if isinstance(user_graph, dict) else str(user_graph.shape),
     "用户关系结构", "以用户ID为键的嵌套关系数据（具体语义待确认）", "图结构分析、GNN研究"]
], columns=["File", "Shape", "Type", "Description", "Usage"])

print(f"Dataset: {data_dir}")
print(f"Users: {inter['userID'].nunique():,} | Items: {inter['itemID'].nunique():,} | Interactions: {len(inter):,}")
display(summary)

# 1. Retrieval

## 1.1 ItemCF

In [ ]:
## 离线

import time
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, diags

# 1. Dataset split
train = inter.loc[inter["x_label"] == 0].copy()
valid = inter.loc[inter["x_label"] == 1].copy()
test = inter.loc[inter["x_label"] == 2].copy()

n_users = int(inter["userID"].max()) + 1
n_items = int(inter["itemID"].max()) + 1

# 2. Experiment configurations
experiments = {
    f"itemcf-{mode}-nb{nb}": {"mode": mode, "nb": nb}
    for mode in ["binary", "rating"]
    for nb in [10, 50, 100, "all"]
}

# 3. Build interaction matrix
def build_matrix(df, mode):
    weights = (
        np.ones(len(df), dtype=np.float32)
        if mode == "binary"
        else df["rating"].to_numpy(dtype=np.float32)
    )
    return csr_matrix(
        (weights, (df["userID"], df["itemID"])),
        shape=(n_users, n_items),
        dtype=np.float32
    )

# 4. Compute cosine similarity and Top-N neighbors
def build_item_index(R, nb):
    norms = np.sqrt(np.asarray(R.power(2).sum(axis=0)).ravel())
    inv_norms = np.divide(
        1.0, norms,
        out=np.zeros_like(norms),
        where=norms > 0
    )

    X = R @ diags(inv_norms)
    S = (X.T @ X).tocsr()
    S.setdiag(0)
    S.eliminate_zeros()
    S.sort_indices()

    if nb == "all":
        return S

    rows, cols, values = [], [], []

    for i in range(n_items):
        start, end = S.indptr[i], S.indptr[i + 1]
        ids = S.indices[start:end]
        scores = S.data[start:end]

        if len(scores) > nb:
            keep = np.argpartition(-scores, nb - 1)[:nb]
            ids, scores = ids[keep], scores[keep]

        order = np.lexsort((ids, -scores))
        rows.extend([i] * len(order))
        cols.extend(ids[order])
        values.extend(scores[order])

    return csr_matrix(
        (np.asarray(values, dtype=np.float32), (rows, cols)),
        shape=(n_items, n_items)
    )

# 5. Offline training
offline_models = {}
offline_records = []

for name, cfg in experiments.items():
    start = time.perf_counter()

    R = build_matrix(train, cfg["mode"])
    S = build_item_index(R, cfg["nb"])

    elapsed = time.perf_counter() - start

    offline_models[name] = {"R": R, "S": S}
    offline_records.append({
        "Experiment": name,
        "OfflineTime(s)": round(elapsed, 3),
        "IndexEdges": S.nnz,
        "IndexMemory(MB)": round(
            (S.data.nbytes + S.indices.nbytes + S.indptr.nbytes) / 1024**2, 3
        )
    })

offline_df = pd.DataFrame(offline_records)

print(f"Users: {n_users:,} | Items: {n_items:,}")
print(f"Train: {len(train):,} | Validation: {len(valid):,} | Test: {len(test):,}")
print("\nOffline computation completed:")
display(offline_df)

In [ ]:
import time
import numpy as np
import pandas as pd
from pathlib import Path

RECALL_TOPK = 100
RECALL_DIR = RECALL_ROOT
RECALL_DIR.mkdir(parents=True, exist_ok=True)

# 1. Online retrieval
def retrieve_itemcf(R, S, topk=100):
    recommendations = {}
    scored_recommendations = {}
    candidate_counts = {}
    start = time.perf_counter()

    for u in range(R.shape[0]):
        history = R.getrow(u)
        scores = np.asarray((history @ S).toarray()).ravel()
        scores[history.indices] = -np.inf

        candidates = np.flatnonzero((scores > 0) & np.isfinite(scores))
        candidate_counts[u] = len(candidates)

        if len(candidates) > topk:
            chosen = np.argpartition(-scores[candidates], topk - 1)[:topk]
            candidates = candidates[chosen]

        order = np.lexsort((candidates, -scores[candidates]))
        ranked = candidates[order]

        recommendations[u] = ranked.tolist()
        scored_recommendations[u] = [
            (int(i), float(scores[i])) for i in ranked
        ]

    elapsed = time.perf_counter() - start
    return recommendations, scored_recommendations, candidate_counts, elapsed

# 2. Run experiments and save recall results
online_results = DiskResults()
scored_results = DiskResults()
online_records = []

for name, model in offline_models.items():
    recs, scored_recs, counts, elapsed = retrieve_itemcf(
        model["R"], model["S"], topk=RECALL_TOPK
    )

    online_results[name] = recs
    scored_results[name] = scored_recs

    rows = [
        (u, item, score, rank, name)
        for u, items in scored_recs.items()
        for rank, (item, score) in enumerate(items, start=1)
    ]

    recall_df = pd.DataFrame(
        rows,
        columns=["userID", "itemID", "score", "rank", "channel"]
    )

    recall_df.to_parquet(
        RECALL_DIR / f"{name}.parquet",
        index=False
    )

    online_records.append({
        "Experiment": name,
        "OnlineTime(s)": round(elapsed, 3),
        "Latency(ms/user)": round(elapsed / n_users * 1000, 4),
        "Users/s": round(n_users / elapsed, 2),
        "AvgCandidates": round(np.mean(list(counts.values())), 2),
        "UsersWith100": sum(c >= RECALL_TOPK for c in counts.values()),
        "EmptyUsers": sum(c == 0 for c in counts.values())
    })

online_df = pd.DataFrame(online_records)

print(f"ItemCF online retrieval completed | Top-{RECALL_TOPK}")
print(f"Saved to: {RECALL_DIR}")
display(online_df)

# 3. Example recommendations
example_user = 0
example_name = "itemcf-rating-nb50"

print(f"\nExperiment: {example_name}")
print(f"User: {example_user}")
print(f"History: {offline_models[example_name]['R'].getrow(example_user).indices.tolist()}")
print(f"Top-20: {scored_results[example_name][example_user][:20]}")

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

EVAL_K = [5, 10, 20, 50, 100]

# 1. Build ground truth
def build_ground_truth(df):
    return {
        int(u): set(items)
        for u, items in df.groupby("userID")["itemID"]
    }

valid_truth = build_ground_truth(valid)
test_truth = build_ground_truth(test)

# 2. Training popularity for novelty
item_pop = np.bincount(
    train["itemID"].to_numpy(dtype=int),
    minlength=n_items
)
item_prob = (item_pop + 1) / (n_users + 1)
item_novelty = -np.log2(item_prob)

# 3. Ranking metrics
def evaluate(recommendations, ground_truth, ks=EVAL_K):
    records = []
    users = sorted(ground_truth)
    catalog = set(np.flatnonzero(item_pop > 0))
    catalog_size = len(catalog)

    for k in ks:
        recalls, ndcgs, precisions = [], [], []
        hitrates, maps, mrrs, novelties = [], [], [], []
        covered_items = set()

        for u in users:
            truth = ground_truth[u]
            recs = recommendations.get(u, [])[:k]

            if not truth:
                continue

            hits = np.array([int(i in truth) for i in recs])
            n_hits = int(hits.sum())
            positions = np.arange(1, len(recs) + 1)

            # Recall and Precision
            recalls.append(n_hits / len(truth))
            precisions.append(n_hits / k)
            hitrates.append(float(n_hits > 0))

            # NDCG
            dcg = np.sum(hits / np.log2(positions + 1))
            ideal_len = min(len(truth), k)
            idcg = np.sum(
                1 / np.log2(np.arange(2, ideal_len + 2))
            )
            ndcgs.append(dcg / idcg)

            # MAP
            cumulative_hits = np.cumsum(hits)
            ap = np.sum(
                (cumulative_hits / positions) * hits
            ) / min(len(truth), k)
            maps.append(ap)

            # MRR
            hit_positions = np.flatnonzero(hits)
            mrrs.append(
                1 / (hit_positions[0] + 1)
                if len(hit_positions) else 0.0
            )

            # Coverage and Novelty
            covered_items.update(recs)
            if recs:
                novelties.append(np.mean(item_novelty[recs]))

        records.append({
            "K": k,
            "Recall": np.mean(recalls),
            "NDCG": np.mean(ndcgs),
            "Precision": np.mean(precisions),
            "HitRate": np.mean(hitrates),
            "MAP": np.mean(maps),
            "MRR": np.mean(mrrs),
            "Coverage": len(covered_items & catalog) / catalog_size,
            "Novelty": np.mean(novelties) if novelties else np.nan
        })

    return pd.DataFrame(records)

# 4. Evaluate all experiments
results = []

for name, recs in online_results.items():
    for split_name, truth in [
        ("Validation", valid_truth),
        ("Test", test_truth)
    ]:
        metrics = evaluate(recs, truth)
        metrics.insert(0, "Split", split_name)
        metrics.insert(0, "Experiment", name)
        results.append(metrics)

results_df = pd.concat(results, ignore_index=True)

# 5. Combine accuracy and efficiency
results_df = (
    results_df
    .merge(offline_df, on="Experiment", how="left")
    .merge(online_df, on="Experiment", how="left")
)

# 6. Save results
output_dir = RESULTS_ROOT
output_dir.mkdir(parents=True, exist_ok=True)

results_df.to_csv(
    output_dir / "itemcf_all_metrics.csv",
    index=False
)

# 7. Show validation comparison
core_metrics = [
    "Experiment", "Recall", "NDCG", "Precision",
    "HitRate", "MAP", "MRR", "Coverage", "Novelty"
]

validation_20 = (
    results_df[
        (results_df["Split"] == "Validation") &
        (results_df["K"] == 20)
    ]
    .sort_values("Recall", ascending=False)
)

print("ItemCF | Validation | Top-20")
display(validation_20[core_metrics].round(4))

print("\nBest configuration by Validation Recall@20:")
best_name = validation_20.iloc[0]["Experiment"]
print(best_name)

print("\nItemCF | Test | Top-20 (Validation-selected model)")
best_test = results_df[
    (results_df["Split"] == "Test") &
    (results_df["K"] == 20) &
    (results_df["Experiment"] == best_name)
]
display(best_test[core_metrics].round(4))

print(f"\nSaved: {output_dir / 'itemcf_all_metrics.csv'}")

Experiment,Split,K,Recall,NDCG,Precision,HitRate,MAP,MRR,Coverage,Novelty,OfflineTime(s),IndexEdges,IndexMemory(MB),OnlineTime(s),Latency(ms/user),Users/s,AvgCandidates,UsersWith100,EmptyUsers
itemcf-binary-nb10,Validation,5,0.012732,0.009003,0.002756,0.013731,0.007595,0.008071,0.791259,11.581494,0.231,70117,0.562,3.778,0.1943,5146.53,55.29,1917,0
itemcf-binary-nb10,Validation,10,0.018388,0.010843,0.001990,0.019851,0.008332,0.008876,0.907336,11.503233,0.231,70117,0.562,3.778,0.1943,5146.53,55.29,1917,0
itemcf-binary-nb10,Validation,20,0.028701,0.013491,0.001543,0.030496,0.009063,0.009621,0.964950,11.423875,0.231,70117,0.562,3.778,0.1943,5146.53,55.29,1917,0
itemcf-binary-nb10,Validation,50,0.045212,0.016863,0.000977,0.047981,0.009616,0.010196,0.986235,11.329671,0.231,70117,0.562,3.778,0.1943,5146.53,55.29,1917,0
itemcf-binary-nb10,Validation,100,0.050475,0.017767,0.000554,0.053741,0.009698,0.010282,0.986377,11.302244,0.231,70117,0.562,3.778,0.1943,5146.53,55.29,1917,0
itemcf-binary-nb10,Test,5,0.012687,0.008859,0.002870,0.014245,0.007344,0.008072,0.791259,11.581494,0.231,70117,0.562,3.778,0.1943,5146.53,55.29,1917,0
itemcf-binary-nb10,Test,10,0.018467,0.010750,0.002088,0.020571,0.008099,0.008908,0.907336,11.503233,0.231,70117,0.562,3.778,0.1943,5146.53,55.29,1917,0
itemcf-binary-nb10,Test,20,0.026981,0.012922,0.001502,0.029571,0.008681,0.009521,0.964950,11.423875,0.231,70117,0.562,3.778,0.1943,5146.53,55.29,1917,0
itemcf-binary-nb10,Test,50,0.043471,0.016341,0.000989,0.048341,0.009228,0.010131,0.986235,11.329671,0.231,70117,0.562,3.778,0.1943,5146.53,55.29,1917,0
itemcf-binary-nb10,Test,100,0.049169,0.017398,0.000587,0.056313,0.009321,0.010249,0.986377,11.302244,0.231,70117,0.562,3.778,0.1943,5146.53,55.29,1917,0


## 1.2 Swing

In [ ]:
from collections import defaultdict
from itertools import combinations
from scipy.sparse import csr_matrix
import numpy as np
import pandas as pd
import time

SWING_ALPHA = 1.0
SWING_NEIGHBORS = [10, 50, 100]
SWING_MODES = ["binary", "rating"]

# 1. Build binary interaction matrix
R_binary = build_matrix(train, "binary").tocsr()
R_binary.sort_indices()

# 2. Compute Swing similarity
def build_swing_similarity(R, alpha=1.0):
    item_scores = defaultdict(float)
    user_items = [
        R.indices[R.indptr[u]:R.indptr[u + 1]]
        for u in range(R.shape[0])
    ]

    item_users = R.T.tocsr()
    user_pairs = set()

    for i in range(R.shape[1]):
        users = item_users.indices[
            item_users.indptr[i]:item_users.indptr[i + 1]
        ]
        user_pairs.update(combinations(users, 2))

    for u, v in user_pairs:
        common = np.intersect1d(
            user_items[u], user_items[v],
            assume_unique=True
        )

        if len(common) < 2:
            continue

        weight = 1.0 / (alpha + len(common))

        for i, j in combinations(common, 2):
            item_scores[(int(i), int(j))] += weight

    rows, cols, values = [], [], []

    for (i, j), score in item_scores.items():
        rows.extend([i, j])
        cols.extend([j, i])
        values.extend([score, score])

    return csr_matrix(
        (np.asarray(values, dtype=np.float32), (rows, cols)),
        shape=(R.shape[1], R.shape[1])
    )

# 3. Keep Top-N item neighbors
def truncate_neighbors(S, nb):
    rows, cols, values = [], [], []

    for i in range(S.shape[0]):
        start, end = S.indptr[i], S.indptr[i + 1]
        ids = S.indices[start:end]
        scores = S.data[start:end]

        if len(scores) > nb:
            keep = np.argpartition(-scores, nb - 1)[:nb]
            ids, scores = ids[keep], scores[keep]

        order = np.lexsort((ids, -scores))
        rows.extend([i] * len(order))
        cols.extend(ids[order])
        values.extend(scores[order])

    return csr_matrix(
        (np.asarray(values, dtype=np.float32), (rows, cols)),
        shape=S.shape
    )

# 4. Build offline indexes
offline_models = {}
offline_records = []

start = time.perf_counter()
S_full = build_swing_similarity(R_binary, SWING_ALPHA)
similarity_time = time.perf_counter() - start

for nb in SWING_NEIGHBORS:
    start = time.perf_counter()
    S = truncate_neighbors(S_full, nb)
    index_time = time.perf_counter() - start

    for mode in SWING_MODES:
        name = f"swing-{mode}-a{SWING_ALPHA:g}-nb{nb}"
        R = build_matrix(train, mode)

        offline_models[name] = {"R": R, "S": S}
        offline_records.append({
            "Experiment": name,
            "OfflineTime(s)": round(similarity_time + index_time, 3),
            "IndexEdges": S.nnz,
            "IndexMemory(MB)": round(
                (S.data.nbytes + S.indices.nbytes + S.indptr.nbytes) / 1024**2, 3
            )
        })

offline_df = pd.DataFrame(offline_records)

print(f"Swing similarity computation: {similarity_time:.2f}s")
print(f"Non-zero similarities: {S_full.nnz:,}")
display(offline_df)

In [ ]:
import time
import numpy as np
import pandas as pd
from pathlib import Path

RECALL_TOPK = 100
RECALL_DIR = RECALL_ROOT
RECALL_DIR.mkdir(parents=True, exist_ok=True)

# 1. Online retrieval and save candidates
online_results = DiskResults()
scored_results = DiskResults()
online_records = []

for name, model in offline_models.items():
    recs, scored_recs, counts, elapsed = retrieve_itemcf(
        model["R"], model["S"], topk=RECALL_TOPK
    )

    online_results[name] = recs
    scored_results[name] = scored_recs

    rows = [
        (u, item, score, rank, name)
        for u, items in scored_recs.items()
        for rank, (item, score) in enumerate(items, start=1)
    ]

    recall_df = pd.DataFrame(
        rows,
        columns=["userID", "itemID", "score", "rank", "channel"]
    )

    recall_df.to_parquet(
        RECALL_DIR / f"{name}.parquet",
        index=False
    )

    online_records.append({
        "Experiment": name,
        "OnlineTime(s)": round(elapsed, 3),
        "Latency(ms/user)": round(elapsed / n_users * 1000, 4),
        "Users/s": round(n_users / elapsed, 2),
        "AvgCandidates": round(np.mean(list(counts.values())), 2),
        "UsersWith100": sum(c >= RECALL_TOPK for c in counts.values()),
        "EmptyUsers": sum(c == 0 for c in counts.values())
    })

online_df = pd.DataFrame(online_records)

print(f"Swing online retrieval completed | Top-{RECALL_TOPK}")
print(f"Saved to: {RECALL_DIR}")
display(online_df)

In [ ]:
from pathlib import Path

results = []

for name, recs in online_results.items():
    for split_name, truth in [
        ("Validation", valid_truth),
        ("Test", test_truth)
    ]:
        metrics = evaluate(recs, truth)
        metrics.insert(0, "Split", split_name)
        metrics.insert(0, "Experiment", name)
        results.append(metrics)

swing_results = pd.concat(results, ignore_index=True)

swing_results = (
    swing_results
    .merge(offline_df, on="Experiment", how="left")
    .merge(online_df, on="Experiment", how="left")
)

output_dir = RESULTS_ROOT
output_dir.mkdir(parents=True, exist_ok=True)

swing_results.to_csv(
    output_dir / "swing_all_metrics.csv",
    index=False
)

validation_20 = (
    swing_results[
        (swing_results["Split"] == "Validation") &
        (swing_results["K"] == 20)
    ]
    .sort_values("Recall", ascending=False)
)

print("Swing | Validation | Top-20")
display(validation_20[[
    "Experiment", "Recall", "NDCG", "Precision",
    "HitRate", "MAP", "MRR", "Coverage", "Novelty"
]].round(4))

best_name = validation_20.iloc[0]["Experiment"]
print(f"\nBest Swing configuration: {best_name}")

print("\nTest | Top-20")
display(
    swing_results[
        (swing_results["Split"] == "Test") &
        (swing_results["K"] == 20) &
        (swing_results["Experiment"] == best_name)
    ][[
        "Experiment", "Recall", "NDCG", "Precision",
        "HitRate", "MAP", "MRR", "Coverage", "Novelty"
    ]].round(4)
)

print(f"\nSaved: {output_dir / 'swing_all_metrics.csv'}")

Experiment,Split,K,Recall,NDCG,Precision,HitRate,MAP,MRR,Coverage,Novelty,OfflineTime(s),IndexEdges,IndexMemory(MB),OnlineTime(s),Latency(ms/user),Users/s,AvgCandidates,UsersWith100,EmptyUsers
swing-binary-a1-nb10,Validation,5,0.035815,0.024548,0.007457,0.037285,0.020576,0.021383,0.208883,6.643326,20.070,24430,0.213,3.048,0.1567,6380.41,33.38,283,32
swing-binary-a1-nb10,Validation,10,0.053684,0.030367,0.005642,0.056313,0.022923,0.023879,0.295587,6.874794,20.070,24430,0.213,3.048,0.1567,6380.41,33.38,283,32
swing-binary-a1-nb10,Validation,20,0.080802,0.037307,0.004281,0.085112,0.024796,0.025857,0.375763,7.188712,20.070,24430,0.213,3.048,0.1567,6380.41,33.38,283,32
swing-binary-a1-nb10,Validation,50,0.102472,0.041879,0.002223,0.109231,0.025567,0.026696,0.389811,7.444949,20.070,24430,0.213,3.048,0.1567,6380.41,33.38,283,32
swing-binary-a1-nb10,Validation,100,0.105703,0.042516,0.001175,0.113242,0.025631,0.026759,0.389953,7.479906,20.070,24430,0.213,3.048,0.1567,6380.41,33.38,283,32
swing-binary-a1-nb10,Test,5,0.034905,0.023914,0.007735,0.038416,0.019664,0.021352,0.208883,6.643326,20.070,24430,0.213,3.048,0.1567,6380.41,33.38,283,32
swing-binary-a1-nb10,Test,10,0.052372,0.029676,0.005842,0.057547,0.021994,0.023878,0.295587,6.874794,20.070,24430,0.213,3.048,0.1567,6380.41,33.38,283,32
swing-binary-a1-nb10,Test,20,0.077018,0.036098,0.004346,0.085215,0.023716,0.025783,0.375763,7.188712,20.070,24430,0.213,3.048,0.1567,6380.41,33.38,283,32
swing-binary-a1-nb10,Test,50,0.101293,0.041341,0.002366,0.114271,0.024593,0.026784,0.389811,7.444949,20.070,24430,0.213,3.048,0.1567,6380.41,33.38,283,32
swing-binary-a1-nb10,Test,100,0.105141,0.042140,0.001271,0.120597,0.024665,0.026882,0.389953,7.479906,20.070,24430,0.213,3.048,0.1567,6380.41,33.38,283,32


## 1.3 UserCF

In [ ]:

import time
import numpy as np
import pandas as pd
from scipy.sparse import diags, csr_matrix

# 6组实验：2种相似度 × 3种相似用户数量
USERCF_NEIGHBORS = [10, 50, 100]
USERCF_METHODS = ["cosine", "pop-penalty"]

def topk_sparse_rows(S, k):
    S = S.tocsr()
    rows, cols, vals = [], [], []
    for u in range(S.shape[0]):
        start, end = S.indptr[u], S.indptr[u + 1]
        idx, data = S.indices[start:end], S.data[start:end]
        if len(data) > k:
            keep = np.argpartition(data, -k)[-k:]
            idx, data = idx[keep], data[keep]
        rows.extend([u] * len(idx))
        cols.extend(idx.tolist())
        vals.extend(data.tolist())
    return csr_matrix((vals, (rows, cols)), shape=S.shape)

def build_usercf_similarity(R, method):
    user_count = np.asarray(R.sum(axis=1)).ravel()
    item_count = np.asarray(R.sum(axis=0)).ravel()
    norm = np.zeros_like(user_count, dtype=float)
    np.divide(1, np.sqrt(user_count), out=norm, where=user_count > 0)
    X = diags(norm) @ R

    if method == "cosine":
        S = (X @ X.T).tocsr()
    else:
        weights = np.zeros_like(item_count, dtype=float)
        mask = item_count > 0
        weights[mask] = 1 / np.log1p(item_count[mask])
        S = (X @ diags(weights) @ X.T).tocsr()

    S.setdiag(0)
    S.eliminate_zeros()
    return S

R_usercf = build_matrix(train, "binary").tocsr()
offline_models = {}
offline_rows = []

for method in USERCF_METHODS:
    start = time.perf_counter()
    S_full = build_usercf_similarity(R_usercf, method)

    for nb in USERCF_NEIGHBORS:
        name = f"usercf-{method}-nb{nb}"
        S = topk_sparse_rows(S_full, nb)
        elapsed = time.perf_counter() - start

        offline_models[name] = {"R": R_usercf, "S": S}
        offline_rows.append({
            "Experiment": name,
            "OfflineTime(s)": elapsed,
            "IndexEdges": S.nnz,
            "IndexMemory(MB)": (
                S.data.nbytes + S.indices.nbytes + S.indptr.nbytes
            ) / 1024**2
        })

offline_df = pd.DataFrame(offline_rows)
display(offline_df)


In [ ]:

import time
from pathlib import Path

RECALL_DIR = RECALL_ROOT
RECALL_DIR.mkdir(parents=True, exist_ok=True)

def retrieve_usercf(R, S, topk=100):
    recommendations, scored_recommendations, candidate_counts = {}, {}, {}
    start = time.perf_counter()

    for u in range(R.shape[0]):
        scores = np.asarray((S.getrow(u) @ R).toarray()).ravel()
        seen = R.indices[R.indptr[u]:R.indptr[u + 1]]
        scores[seen] = -np.inf

        candidates = np.flatnonzero((scores > 0) & np.isfinite(scores))
        candidate_counts[u] = len(candidates)

        if len(candidates):
            order = np.lexsort((candidates, -scores[candidates]))
            items = candidates[order[:topk]]
        else:
            items = np.array([], dtype=int)

        recommendations[u] = items.tolist()
        scored_recommendations[u] = [
            (int(i), float(scores[i])) for i in items
        ]

    elapsed = time.perf_counter() - start
    return recommendations, scored_recommendations, candidate_counts, elapsed

online_results = DiskResults()
scored_results = DiskResults()
online_rows = []

for name, model in offline_models.items():
    recs, scored_recs, counts, elapsed = retrieve_usercf(
        model["R"], model["S"], topk=100
    )

    online_results[name] = recs
    scored_results[name] = scored_recs

    records = [
        (u, item, score, rank, name)
        for u, pairs in scored_recs.items()
        for rank, (item, score) in enumerate(pairs, 1)
    ]
    pd.DataFrame(
        records,
        columns=["userID", "itemID", "score", "rank", "channel"]
    ).to_parquet(RECALL_DIR / f"{name}.parquet", index=False)

    counts_array = np.array(list(counts.values()))
    online_rows.append({
        "Experiment": name,
        "OnlineTime(s)": elapsed,
        "Latency(ms/user)": elapsed * 1000 / len(recs),
        "Users/s": len(recs) / elapsed if elapsed > 0 else np.nan,
        "AvgCandidates": counts_array.mean(),
        "UsersWith100": int((counts_array >= 100).sum()),
        "EmptyUsers": int((counts_array == 0).sum())
    })

online_df = pd.DataFrame(online_rows)
display(online_df)

example_name = "usercf-pop-penalty-nb50"
print("Example user 0:", scored_results[example_name].get(0, [])[:10])
print("Saved to:", RECALL_DIR)


In [ ]:

from pathlib import Path

# 复用 ItemCF Evaluation Cell 中的 evaluate() 和 EVAL_K
results_dir = RESULTS_ROOT
results_dir.mkdir(parents=True, exist_ok=True)

valid_truth = (
    inter.loc[inter["x_label"] == 1]
    .groupby("userID")["itemID"]
    .apply(set).to_dict()
)
test_truth = (
    inter.loc[inter["x_label"] == 2]
    .groupby("userID")["itemID"]
    .apply(set).to_dict()
)

metric_rows = []

for name, recs in online_results.items():
    for split, truth in [("Validation", valid_truth), ("Test", test_truth)]:
        metrics = evaluate(recs, truth, ks=EVAL_K)

        if isinstance(metrics, pd.DataFrame):
            frame = metrics.copy()
            frame["Experiment"] = name
            frame["Split"] = split
            metric_rows.extend(frame.to_dict("records"))
        else:
            for k, values in metrics.items():
                metric_rows.append({
                    "Experiment": name,
                    "Split": split,
                    "K": k,
                    **values
                })

metrics_df = pd.DataFrame(metric_rows)
metrics_df = metrics_df.merge(offline_df, on="Experiment", how="left")
metrics_df = metrics_df.merge(online_df, on="Experiment", how="left")

metrics_df.to_csv(
    results_dir / "usercf_all_metrics.csv", index=False
)

valid20 = metrics_df[
    (metrics_df["Split"] == "Validation") &
    (metrics_df["K"] == 20)
].sort_values("Recall", ascending=False)

display(valid20)
best_name = valid20.iloc[0]["Experiment"]
print("Best experiment:", best_name)

display(metrics_df[
    (metrics_df["Experiment"] == best_name) &
    (metrics_df["Split"] == "Test") &
    (metrics_df["K"] == 20)
])
print("Saved to:", results_dir / "usercf_all_metrics.csv")


K,Recall,NDCG,Precision,HitRate,MAP,MRR,Coverage,Novelty,Experiment,Split,OfflineTime(s),IndexEdges,IndexMemory(MB),OnlineTime(s),Latency(ms/user),Users/s,AvgCandidates,UsersWith100,EmptyUsers
5,0.011859,0.008357,0.002510,0.012548,0.007103,0.007399,0.846459,8.821677,usercf-cosine-nb10,Validation,0.381434,194441,2.299381,5.101392,0.262350,3811.704777,26.970429,10,0
10,0.018715,0.010575,0.001985,0.019748,0.007997,0.008331,0.954874,8.942071,usercf-cosine-nb10,Validation,0.381434,194441,2.299381,5.101392,0.262350,3811.704777,26.970429,10,0
20,0.030640,0.013615,0.001612,0.032091,0.008827,0.009187,0.989783,9.114608,usercf-cosine-nb10,Validation,0.381434,194441,2.299381,5.101392,0.262350,3811.704777,26.970429,10,0
50,0.039267,0.015416,0.000835,0.041399,0.009134,0.009516,0.995459,9.180283,usercf-cosine-nb10,Validation,0.381434,194441,2.299381,5.101392,0.262350,3811.704777,26.970429,10,0
100,0.040046,0.015556,0.000428,0.042170,0.009149,0.009529,0.995601,9.181872,usercf-cosine-nb10,Validation,0.381434,194441,2.299381,5.101392,0.262350,3811.704777,26.970429,10,0
5,0.013381,0.009172,0.003034,0.015068,0.007488,0.008365,0.846459,8.821677,usercf-cosine-nb10,Test,0.381434,194441,2.299381,5.101392,0.262350,3811.704777,26.970429,10,0
10,0.020988,0.011636,0.002355,0.023399,0.008464,0.009436,0.954874,8.942071,usercf-cosine-nb10,Test,0.381434,194441,2.299381,5.101392,0.262350,3811.704777,26.970429,10,0
20,0.034268,0.015047,0.001911,0.037902,0.009373,0.010426,0.989783,9.114608,usercf-cosine-nb10,Test,0.381434,194441,2.299381,5.101392,0.262350,3811.704777,26.970429,10,0
50,0.042706,0.016880,0.000977,0.047879,0.009688,0.010787,0.995459,9.180283,usercf-cosine-nb10,Test,0.381434,194441,2.299381,5.101392,0.262350,3811.704777,26.970429,10,0
100,0.042992,0.016943,0.000496,0.048239,0.009695,0.010793,0.995601,9.181872,usercf-cosine-nb10,Test,0.381434,194441,2.299381,5.101392,0.262350,3811.704777,26.970429,10,0


## 1.4 Dual-Tower Model

## DualTower Experiments

基于 Amazon Baby 数据集构建双塔召回模型（DualTower），采用 User Tower 和 Item Tower 分别生成 64 维向量，通过相似度完成 Top-100 召回。所有负样本均来自 Batch 内其他商品。

### 1. Loss Function Comparison

固定 Item Tower 使用全部特征（Item ID + Text + Image），比较不同损失函数及训练策略。

| 实验 | 目的 |
|---|---|
| Pointwise BCE | 独立预测正负样本 |
| Pairwise Triplet | 优化正负样本间隔 |
| Pairwise BPR | 优化正负样本相对排序 |
| Listwise Softmax | Batch 内多商品竞争 |
| Listwise Batch 1024 | 比较更多负样本的效果 |
| Listwise Bias-corrected | 检验采样偏差修正的效果 |

### 2. Feature Ablation

固定使用 Listwise + In-batch Softmax，比较不同 Item Tower 特征组合，并分别测试普通版与纠偏版。

| 特征组合 | 普通版 | 纠偏版 |
|---|:---:|:---:|
| Item ID | ✓ | ✓ |
| Item ID + Text | ✓ | ✓ |
| Item ID + Image | ✓ | ✓ |
| Item ID + Text + Image | ✓ | ✓ |

两组实验共享 2 个配置，因此共 **12 组独立实验**。

### 3. Evaluation & Results

统一评估 Recall、NDCG、Precision、HitRate、MAP、MRR、Coverage、Novelty，以及训练和召回效率。

- `recall/dualtower-*.parquet`：每组实验的 Top-100 召回候选及得分，供后续 Ranking 使用。
- `results/dualtower_all_metrics.csv`：全部实验的评估指标。

使用 Validation Recall@20 选择最佳模型，并报告对应的 Test 结果。

In [ ]:
# DualTower Offline：12 组实验；全部负样本仅来自当前 batch，禁止额外采样
import time, gc
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset
from scipy.sparse import csr_matrix

SEED, DIM, EPOCHS, LR, TEMP, MARGIN = 42, 64, 10, 1e-3, 0.1, 0.2
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(SEED); np.random.seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

# 直接复用 Data Loading Cell 的 inter、text_feat、image_feat、data_dir
train = inter.loc[inter.x_label == 0, ['userID', 'itemID']].copy()
valid = inter.loc[inter.x_label == 1, ['userID', 'itemID']].copy()
test = inter.loc[inter.x_label == 2, ['userID', 'itemID']].copy()
N_USERS = int(inter.userID.max()) + 1
N_ITEMS = int(inter.itemID.max()) + 1
assert len(text_feat) == len(image_feat) == N_ITEMS

# 仅用训练集构造已知正样本，避免 Batch 内把已知正例错误当负例
TRAIN_CSR = csr_matrix((np.ones(len(train), dtype=np.float32),
    (train.userID.to_numpy(), train.itemID.to_numpy())), shape=(N_USERS, N_ITEMS))
TRAIN_CSR.data[:] = 1
TRAIN_CSR.sort_indices()
train_pairs = torch.tensor(train[['userID', 'itemID']].to_numpy(), dtype=torch.long)
item_freq = np.asarray(TRAIN_CSR.sum(axis=0)).ravel().astype(np.float64)
item_prob = np.maximum(item_freq / max(item_freq.sum(), 1), 1e-12)
LOG_ITEM_PROB = torch.tensor(np.log(item_prob), dtype=torch.float32, device=DEVICE)

# 预训练的文本/图像向量先做 L2 归一化，再进入可训练线性投影
TEXT = torch.tensor(np.array(text_feat, dtype=np.float32), dtype=torch.float32)
IMAGE = torch.tensor(np.array(image_feat, dtype=np.float32), dtype=torch.float32)
TEXT = F.normalize(TEXT, dim=1)
IMAGE = F.normalize(IMAGE, dim=1)

class DualTower(nn.Module):
    def __init__(self, features):
        super().__init__()
        self.features = features
        self.user_emb = nn.Embedding(N_USERS, DIM)
        self.item_emb = nn.Embedding(N_ITEMS, DIM)
        self.text_proj = nn.Linear(TEXT.shape[1], DIM) if 'text' in features else None
        self.image_proj = nn.Linear(IMAGE.shape[1], DIM) if 'image' in features else None
        self.user_mlp = nn.Sequential(nn.Linear(DIM, 128), nn.ReLU(), nn.Linear(128, DIM))
        n_features = 1 + int('text' in features) + int('image' in features)
        self.item_mlp = nn.Sequential(nn.Linear(DIM * n_features, 128), nn.ReLU(), nn.Linear(128, DIM))
        self.register_buffer('text_data', TEXT if 'text' in features else torch.empty(0), persistent=False)
        self.register_buffer('image_data', IMAGE if 'image' in features else torch.empty(0), persistent=False)
        nn.init.normal_(self.user_emb.weight, std=0.02)
        nn.init.normal_(self.item_emb.weight, std=0.02)

    def encode_users(self, users):
        return F.normalize(self.user_mlp(self.user_emb(users)), dim=-1)

    def encode_items(self, items):
        # 所有特征在进入 Item MLP 前 concatenate；不用的特征完全不参与
        parts = [self.item_emb(items)]
        if self.text_proj is not None: parts.append(self.text_proj(self.text_data[items]))
        if self.image_proj is not None: parts.append(self.image_proj(self.image_data[items]))
        return F.normalize(self.item_mlp(torch.cat(parts, dim=-1)), dim=-1)

# Loss 比较：全部使用 ID + Text + Image，batch=512（额外比较 batch=1024）
EXPERIMENTS = [
    # Pointwise BCE：batch 对角线为正例，其他有效位置为负例
    dict(name='dualtower-pointwise-bce', loss='bce', features=('text','image'), batch=512, corrected=False),
    # Pairwise Triplet：正例得分至少比 batch 内负例高 margin=0.2
    dict(name='dualtower-pairwise-triplet', loss='triplet', features=('text','image'), batch=512, corrected=False),
    # Pairwise BPR：最大化正例得分高于 batch 内负例的概率
    dict(name='dualtower-pairwise-bpr', loss='bpr', features=('text','image'), batch=512, corrected=False),
    # Listwise 基线：完整 ID + Text + Image，batch=512，不纠偏
    dict(name='dualtower-listwise-inbatch', loss='listwise', features=('text','image'), batch=512, corrected=False),
    # Batch Size 实验：其他不变，batch=1024，增加 batch 内负例数量
    dict(name='dualtower-listwise-inbatch-batch1024', loss='listwise', features=('text','image'), batch=1024, corrected=False),
    # 纠偏实验：完整特征，使用 -log(采样概率) 修正训练 logits
    dict(name='dualtower-listwise-inbatch-biascorrected', loss='listwise', features=('text','image'), batch=512, corrected=True),
    # 特征消融：仅 Item ID，不使用文本或图像
    dict(name='dualtower-listwise-id-only', loss='listwise', features=(), batch=512, corrected=False),
    # 特征消融：Item ID + Text，进入 MLP 前 concatenate
    dict(name='dualtower-listwise-id-text', loss='listwise', features=('text',), batch=512, corrected=False),
    # 特征消融：Item ID + Image，进入 MLP 前 concatenate
    dict(name='dualtower-listwise-id-image', loss='listwise', features=('image',), batch=512, corrected=False),
    # 纠偏消融：仅 Item ID，比较纠偏相对普通 ID-only 的收益
    dict(name='dualtower-listwise-id-only-biascorrected', loss='listwise', features=(), batch=512, corrected=True),
    # 纠偏消融：Item ID + Text，比较文本条件下纠偏是否有效
    dict(name='dualtower-listwise-id-text-biascorrected', loss='listwise', features=('text',), batch=512, corrected=True),
    # 纠偏消融：Item ID + Image，比较图像条件下纠偏是否有效
    dict(name='dualtower-listwise-id-image-biascorrected', loss='listwise', features=('image',), batch=512, corrected=True),
]
assert len(EXPERIMENTS) == len({x['name'] for x in EXPERIMENTS}) == 12

# 训练时屏蔽其他位置的已知正样本及重复 item，防止 false negatives
# 注意：验证/测试的未来交互不能用于此处屏蔽，否则造成标签泄漏
def inbatch_valid_mask(users, items):
    uu = users.detach().cpu().numpy()
    ii = items.detach().cpu().numpy()
    known = TRAIN_CSR[uu][:, ii].toarray().astype(bool)
    same_item = ii[:, None] == ii[None, :]
    mask = ~(known | same_item)
    np.fill_diagonal(mask, True)
    return torch.as_tensor(mask, dtype=torch.bool, device=DEVICE)

def training_loss(model, users, items, cfg):
    uvec = model.encode_users(users)
    ivec = model.encode_items(items)
    scores = uvec @ ivec.T  # [B, B]，每行对角线为正样本
    valid_mask = inbatch_valid_mask(users, items)
    diag = torch.arange(len(users), device=DEVICE)
    if cfg['loss'] == 'listwise':
        # 校正项仅用于训练，线上召回仍使用原始余弦相似度
        logits = scores / TEMP
        if cfg['corrected']:
            logits = logits - LOG_ITEM_PROB[items][None, :]
        logits = logits.masked_fill(~valid_mask, -1e9)
        return F.cross_entropy(logits, diag)
    negative_mask = valid_mask.clone()
    negative_mask[diag, diag] = False
    if cfg['loss'] == 'bce':
        # 为防止负例数量压倒正例，对正负两类分别求均值
        positive = F.softplus(-scores[diag, diag]).mean()
        negative = F.softplus(scores[negative_mask]).mean() if negative_mask.any() else positive * 0
        return 0.5 * (positive + negative)
    # Pairwise：每个正例配一个当前 batch 中有效的随机负例
    valid_rows = negative_mask.any(dim=1)
    if not valid_rows.any(): return scores.sum() * 0
    row_ids = torch.where(valid_rows)[0]
    choices = torch.rand(negative_mask.shape, device=DEVICE).masked_fill(~negative_mask, -1)
    neg_ids = choices[row_ids].argmax(dim=1)
    pos = scores[row_ids, row_ids]
    neg = scores[row_ids, neg_ids]
    if cfg['loss'] == 'triplet': return F.relu(MARGIN + neg - pos).mean()
    return F.softplus(neg - pos).mean()  # BPR

OFFLINE_DIR = RESULTS_ROOT / 'dualtower_models'
OFFLINE_DIR.mkdir(parents=True, exist_ok=True)
offline_models, offline_rows = {}, []
print('Device:', DEVICE, '| Experiments:', len(EXPERIMENTS))
for cfg in EXPERIMENTS:
    torch.manual_seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
    model = DualTower(cfg['features']).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    loader = DataLoader(TensorDataset(train_pairs), batch_size=cfg['batch'], shuffle=True,
                        drop_last=False, pin_memory=DEVICE.type=='cuda')
    t0, epoch_losses = time.perf_counter(), []
    for epoch in range(EPOCHS):
        model.train(); total_loss, total_n = 0., 0
        for (pairs,) in loader:
            pairs = pairs.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            loss = training_loss(model, pairs[:, 0], pairs[:, 1], cfg)
            loss.backward(); optimizer.step()
            total_loss += float(loss.detach()) * len(pairs)
            total_n += len(pairs)
        epoch_losses.append(total_loss / max(total_n, 1))
    offline_seconds = time.perf_counter() - t0
    model.eval()
    with torch.no_grad():
        item_vectors = torch.cat([model.encode_items(idx).cpu() for idx in
            torch.arange(N_ITEMS, device=DEVICE).split(512)], dim=0).numpy().astype('float32')
    # 保存完整模型与 item vectors；之后 Online 可单独重载，不依赖内存模型
    path = OFFLINE_DIR / f"{cfg['name']}.pt"
    torch.save({'config': cfg, 'state_dict': {k:v.cpu() for k,v in model.state_dict().items()},
                'item_vectors': item_vectors, 'epochs': EPOCHS}, path)
    offline_models[cfg['name']] = {'path': str(path), 'config': cfg}
    offline_rows.append({'Experiment':cfg['name'], 'OfflineTime(s)':offline_seconds,
                         'TrainLoss':epoch_losses[-1], 'Epochs':EPOCHS,
                         'IndexEdges':N_ITEMS, 'IndexMemory(MB)':item_vectors.nbytes/2**20})
    print(cfg['name'], '| train loss:', round(epoch_losses[-1], 4),
          '| offline sec:', round(offline_seconds, 1))
    del model, optimizer, item_vectors
    gc.collect()
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()
offline_df = pd.DataFrame(offline_rows)
display(offline_df)


In [ ]:
# DualTower Online：每组独立召回 Top-100，保存带 score/rank/channel 的 Parquet
import time
RECALL_DIR = RECALL_ROOT
RECALL_DIR.mkdir(parents=True, exist_ok=True)
ALL_USER_IDS = np.arange(N_USERS, dtype=np.int64)
TOPK = 100
online_results, online_rows = DiskResults(), []

@torch.no_grad()
def retrieve_dualtower(model, item_vectors, topk=100, user_batch=256):
    model.eval()
    items_gpu = torch.as_tensor(item_vectors, dtype=torch.float32, device=DEVICE)
    records, recs = [], {}
    counts = np.zeros(N_USERS, dtype=np.int32)
    start = time.perf_counter()
    for start_u in range(0, N_USERS, user_batch):
        end_u = min(start_u + user_batch, N_USERS)
        users = torch.arange(start_u, end_u, device=DEVICE)
        scores = (model.encode_users(users) @ items_gpu.T).cpu().numpy()
        for local, u in enumerate(range(start_u, end_u)):
            # 仅过滤训练集已交互物品；不能过滤 validation/test 正例
            lo, hi = TRAIN_CSR.indptr[u:u+2]
            scores[local, TRAIN_CSR.indices[lo:hi]] = -np.inf
            row = scores[local]
            eligible = np.flatnonzero(np.isfinite(row))
            counts[u] = len(eligible)
            k = min(topk, len(eligible))
            if k:
                top = eligible[np.argpartition(-row[eligible], k-1)[:k]]
                top = top[np.lexsort((top, -row[top]))]
                recs[u] = top.tolist()
                records.extend((u, int(item), float(row[item]), rank, cfg_name)
                               for rank, item in enumerate(top, 1))
            else:
                recs[u] = []
    return recs, records, counts, time.perf_counter() - start

for cfg in EXPERIMENTS:
    cfg_name = cfg['name']
    checkpoint = torch.load(OFFLINE_DIR / f'{cfg_name}.pt', map_location='cpu', weights_only=False)
    model = DualTower(cfg['features']).to(DEVICE)
    model.load_state_dict(checkpoint['state_dict'])
    recs, records, counts, elapsed = retrieve_dualtower(model, checkpoint['item_vectors'], TOPK)
    frame = pd.DataFrame(records, columns=['userID','itemID','score','rank','channel'])
    frame.to_parquet(RECALL_DIR / f'{cfg_name}.parquet', index=False)
    online_results[cfg_name] = recs
    online_rows.append({'Experiment':cfg_name, 'OnlineTime(s)':elapsed,
                        'Latency(ms/user)':elapsed*1000/N_USERS,
                        'Users/s':N_USERS/max(elapsed, 1e-9),
                        'AvgCandidates':float(np.minimum(counts, TOPK).mean()),
                        'UsersWith100':int((counts >= TOPK).sum()),
                        'EmptyUsers':int((counts == 0).sum())})
    print(cfg_name, '| rows:', len(frame), '| online sec:', round(elapsed, 2))
    del model, checkpoint, recs, records, frame
    gc.collect()
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()
online_df = pd.DataFrame(online_rows)
display(online_df)


In [ ]:
# DualTower Evaluation：Validation/Test × K=[5,10,20,50,100]，统一保存 CSV
# 复用此前 ItemCF 的 evaluate()（如已定义）；若未定义，使用下面的自包含实现
from collections import defaultdict
EVAL_K = [5, 10, 20, 50, 100]
train_pop = np.asarray(TRAIN_CSR.sum(axis=0)).ravel()
observed_catalog = max(int((train_pop > 0).sum()), 1)
novelty_per_item = -np.log2((train_pop + 1) / (N_USERS + 1))

def dualtower_evaluate(recommendations, truth_df, ks=EVAL_K):
    gt = defaultdict(set)
    for u, i in truth_df[['userID','itemID']].itertuples(index=False, name=None):
        gt[int(u)].add(int(i))
    users = sorted(gt)
    result = []
    for k in ks:
        sums = defaultdict(float); recommended_items = set()
        for u in users:
            positives = gt[u]
            picks = recommendations.get(u, [])[:k]
            hits = np.fromiter((i in positives for i in picks), dtype=np.float64)
            hit_count = float(hits.sum())
            ranks = np.flatnonzero(hits) + 1
            dcg = float((hits / np.log2(np.arange(2, len(hits)+2))).sum())
            ideal_len = min(k, len(positives))
            idcg = float((1 / np.log2(np.arange(2, ideal_len+2))).sum())
            sums['Recall'] += hit_count / len(positives)
            sums['Precision'] += hit_count / k
            sums['NDCG'] += dcg / idcg if idcg else 0
            sums['HitRate'] += float(hit_count > 0)
            sums['MRR'] += 1 / ranks[0] if len(ranks) else 0
            if len(ranks):
                sums['MAP'] += sum(j/r for j, r in enumerate(ranks, 1)) / ideal_len
            if picks:
                sums['Novelty'] += float(novelty_per_item[picks].mean())
                recommended_items.update(picks)
        n = max(len(users), 1)
        row = {'K':k, **{metric:sums[metric]/n for metric in
            ['Recall','NDCG','Precision','HitRate','MAP','MRR','Novelty']}}
        row['Coverage'] = len(recommended_items) / observed_catalog
        result.append(row)
    return pd.DataFrame(result)

metric_rows = []
for cfg in EXPERIMENTS:
    name = cfg['name']
    # 若 Online Cell 已运行，使用内存结果；否则直接从 Parquet 重建
    if name in online_results:
        recs = online_results[name]
    else:
        frame = pd.read_parquet(RECALL_DIR / f'{name}.parquet')
        recs = {int(u):group.sort_values('rank').itemID.astype(int).tolist()
                for u, group in frame.groupby('userID', sort=False)}
    for split, truth in [('Validation', valid), ('Test', test)]:
        scores = dualtower_evaluate(recs, truth)
        scores.insert(0, 'Split', split)
        scores.insert(0, 'Experiment', name)
        metric_rows.append(scores)
metrics_df = pd.concat(metric_rows, ignore_index=True)
metrics_df = metrics_df.merge(offline_df, on='Experiment', how='left').merge(
    online_df, on='Experiment', how='left')
RESULTS_DIR = RESULTS_ROOT
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
metrics_df.to_csv(RESULTS_DIR / 'dualtower_all_metrics.csv', index=False)

# 仅根据 Validation Recall@20 选最优配置，避免利用 Test 调参
leaderboard = metrics_df.query("Split == 'Validation' and K == 20").sort_values(
    ['Recall', 'NDCG'], ascending=False)
display(leaderboard[['Experiment','Recall','NDCG','Precision','HitRate','Coverage','OfflineTime(s)','Latency(ms/user)']])
best = leaderboard.iloc[0]['Experiment']
print('Best on Validation Recall@20:', best)
display(metrics_df.query('Experiment == @best and Split == "Test" and K == 20'))
print('Saved:', RESULTS_DIR / 'dualtower_all_metrics.csv')


Experiment,Split,K,Recall,NDCG,Precision,HitRate,MAP,MRR,Novelty,Coverage,OfflineTime(s),TrainLoss,Epochs,IndexEdges,IndexMemory(MB),OnlineTime(s),Latency(ms/user),Users/s,AvgCandidates,UsersWith100,EmptyUsers
dualtower-pointwise-bce,Validation,5,0.004020,0.002335,0.000833,0.004166,0.001760,0.001840,10.742290,0.999007,17.806646,0.590740,10,7050,1.721191,3.228351,0.166025,6023.199544,100.0,19445,0
dualtower-pointwise-bce,Validation,10,0.007718,0.003522,0.000807,0.008074,0.002230,0.002336,10.743543,1.000142,17.806646,0.590740,10,7050,1.721191,3.228351,0.166025,6023.199544,100.0,19445,0
dualtower-pointwise-bce,Validation,20,0.013643,0.005040,0.000712,0.014194,0.002648,0.002766,10.741304,1.000426,17.806646,0.590740,10,7050,1.721191,3.228351,0.166025,6023.199544,100.0,19445,0
dualtower-pointwise-bce,Validation,50,0.033116,0.008897,0.000699,0.034765,0.003238,0.003387,10.733548,1.000426,17.806646,0.590740,10,7050,1.721191,3.228351,0.166025,6023.199544,100.0,19445,0
dualtower-pointwise-bce,Validation,100,0.060491,0.013359,0.000644,0.063872,0.003613,0.003786,10.723355,1.000426,17.806646,0.590740,10,7050,1.721191,3.228351,0.166025,6023.199544,100.0,19445,0
dualtower-pointwise-bce,Test,5,0.002764,0.001719,0.000627,0.003137,0.001308,0.001521,10.742290,0.999007,17.806646,0.590740,10,7050,1.721191,3.228351,0.166025,6023.199544,100.0,19445,0
dualtower-pointwise-bce,Test,10,0.006248,0.002866,0.000725,0.007251,0.001752,0.002045,10.743543,1.000142,17.806646,0.590740,10,7050,1.721191,3.228351,0.166025,6023.199544,100.0,19445,0
dualtower-pointwise-bce,Test,20,0.012155,0.004391,0.000694,0.013731,0.002162,0.002486,10.741304,1.000426,17.806646,0.590740,10,7050,1.721191,3.228351,0.166025,6023.199544,100.0,19445,0
dualtower-pointwise-bce,Test,50,0.030314,0.008050,0.000689,0.033993,0.002719,0.003107,10.733548,1.000426,17.806646,0.590740,10,7050,1.721191,3.228351,0.166025,6023.199544,100.0,19445,0
dualtower-pointwise-bce,Test,100,0.058265,0.012661,0.000661,0.064592,0.003106,0.003526,10.723355,1.000426,17.806646,0.590740,10,7050,1.721191,3.228351,0.166025,6023.199544,100.0,19445,0


# 2. Pre-Ranking


## Three-Tower Pre-Ranking（三塔粗排）

### 实验目标

研究三塔粗排模型中不同特征交叉方法对推荐效果和在线推理效率的影响。

三塔结构：
- **User Tower**：User ID Embedding、用户历史统计特征，输出 64D 用户向量。
- **Item Tower**：Item ID Embedding、Text Embedding、Image Embedding，输出 64D 商品向量，支持离线缓存。
- **Cross Tower**：输入统计特征和用户—商品匹配特征，分别采用 Linear、MLP、FM、DCN 学习特征交叉。

三个塔的输出经过 Concatenate → Shared MLP → Sigmoid，预测候选商品的交互分数。

### 实验配置

| 实验名称 | Cross Tower | 实验目的 |
|---|---|---|
| `threetower-none` | 无 | 不使用交叉塔的基线 |
| `threetower-linear` | Linear | 一阶线性特征组合 |
| `threetower-mlp` | MLP + ReLU | 隐式非线性交互 |
| `threetower-fm` | FM，k=16 | 显式二阶特征交叉 |
| `threetower-dcn2` | DCN，2 层 | 显式最高三阶交叉 |
| `threetower-dcn3` | DCN，3 层 | 显式最高四阶交叉 |

所有实验固定 User Tower、Item Tower、训练数据和上层预测网络，仅改变 Cross Tower。

### 数据与训练

- **数据集**：Amazon Baby。
- **召回来源**：读取 `Amazon/baby/recall/` 目录下的**全部 Parquet 召回结果**，包含 ItemCF、Swing、UserCF、DualTower 等所有已保存的召回通道及实验版本，不筛选最优版本。
- **候选集构建**：合并全部召回结果，按照 `(userID, itemID)` 去重，并排除训练集中已交互的商品。
- **训练目标**：预测用户与候选商品是否发生正向交互。
- **负样本**：从未观察到交互的商品中采样，不代表真实负反馈。
- **损失函数**：Binary Cross Entropy。
- **优化器**：Adam。
- **Batch Size**：512。
- **Embedding Dimension**：64。
- **Epochs**：最多 30，使用 Early Stopping。
- **模型选择**：Validation NDCG@20。

统计特征和交叉特征仅使用训练阶段允许访问的历史数据，避免标签泄漏。

### 评估指标

- **排序效果**：Recall、NDCG、Precision、HitRate、MAP、MRR。
- **推荐特性**：Coverage、Novelty。
- **离线成本**：训练时间、参数量、模型大小。
- **在线性能**：推理总时间、Latency(ms/user)、Users/s。
- **缓存收益**：Item Tower 缓存前后的推理时间对比。

统一评估 `K = [5, 10, 20, 50, 100]`，以 **NDCG@20** 为主要比较指标。

### 实验输出

- `Amazon/baby/prerank/{experiment}.parquet`：保存各实验的 Top-100 粗排结果，包含 `userID`、`itemID`、`score`、`rank`、`channel`。
- `RecSys-Lab/results/threetower_all_metrics.csv`：汇总六组实验的排序指标和推理性能。

重点比较 **无交叉塔 → FM → DCN** 的效果与计算成本，验证显式特征交叉是否适合粗排阶段。


In [ ]:
# Previous sections have persisted all recall artifacts and metrics.
for _name in ('online_results', 'scored_results', 'offline_models', 'rows',
              'records', 'recall_df', 'recs', 'scored_recs', 'model',
              'R', 'S', 'S_full', 'R_usercf', 'R_binary', 'TEXT', 'IMAGE'):
    globals().pop(_name, None)
gc.collect()
import time, math, random, copy
from pathlib import Path
import numpy as np, pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

SEED = 333
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
data_dir = INPUT_DATA_DIR
inter = pd.read_csv(data_dir/'baby.inter', sep='\t')
image = np.load(data_dir/'image_feat.npy', mmap_mode='r')
text = np.load(data_dir/'text_feat.npy', mmap_mode='r')
assert {'userID','itemID','rating','timestamp','x_label'}.issubset(inter.columns)
N_USERS, N_ITEMS = int(inter.userID.max())+1, len(text)
assert len(image)==N_ITEMS
TRAIN = inter[inter.x_label==0].sort_values(['userID','timestamp','itemID']).copy()
VAL = inter[inter.x_label==1].copy(); TEST = inter[inter.x_label==2].copy()
OUT = WORK/'prerank'; OUT.mkdir(parents=True, exist_ok=True)
RESULTS = RESULTS_ROOT; RESULTS.mkdir(parents=True, exist_ok=True)
print('Device:',DEVICE,'Train/Val/Test:',len(TRAIN),len(VAL),len(TEST))

# 训练内部按用户时间留最后一条作正例；其余作为训练特征历史。
# 这样用户统计/历史内容向量不会包含其训练目标；验证/测试只使用原始 TRAIN。
last = TRAIN.groupby('userID', sort=False).tail(1)
counts = TRAIN.groupby('userID').size()
train_pos = last[last.userID.map(counts)>=2][['userID','itemID']].to_numpy(dtype=np.int64)
HIST_TRAIN = TRAIN.drop(index=train_pos[:,0]*0) if False else TRAIN.drop(index=last[last.userID.map(counts)>=2].index)
HIST_FULL = TRAIN

# 归一化图文向量，避免图像 4096 维数值尺度压倒文本特征。
def normalize_rows(a):
    a = np.asarray(a, dtype=np.float32).copy()
    return a / np.maximum(np.linalg.norm(a,axis=1,keepdims=True),1e-8)
TEXT = normalize_rows(text); IMAGE = normalize_rows(image)

def build_features(history):
    us = history.groupby('userID').agg(n=('itemID','size'), avg=('rating','mean'))
    it = history.groupby('itemID').agg(n=('userID','size'), avg=('rating','mean'), std=('rating','std'))
    u = np.zeros((N_USERS,2),np.float32); i = np.zeros((N_ITEMS,3),np.float32)
    ui=us.index.to_numpy(dtype=int); ii=it.index.to_numpy(dtype=int)
    u[ui,0]=np.log1p(us.n.to_numpy()); u[ui,1]=us.avg.to_numpy()
    i[ii,0]=np.log1p(it.n.to_numpy()); i[ii,1]=it.avg.to_numpy()
    i[ii,2]=it['std'].fillna(0).to_numpy()
    # 用户历史商品图文向量均值（仅历史），用于交叉塔匹配特征。
    uh_t=np.zeros((N_USERS,TEXT.shape[1]),np.float32)
    uh_i=np.zeros((N_USERS,IMAGE.shape[1]),np.float32)
    ct=np.zeros(N_USERS,np.float32)
    uu=history.userID.to_numpy(dtype=int); jj=history.itemID.to_numpy(dtype=int)
    for start in range(0, len(uu), 2048):
        ublock, iblock = uu[start:start+2048], jj[start:start+2048]
        np.add.at(uh_t, ublock, TEXT[iblock])
        np.add.at(uh_i, ublock, IMAGE[iblock])
        np.add.at(ct, ublock, 1)
    uh_t=normalize_rows(uh_t/np.maximum(ct[:,None],1)); uh_i=normalize_rows(uh_i/np.maximum(ct[:,None],1))
    # 统计特征与人工匹配特征是输入；FM/DCN 才负责显式学习交叉。
    return {'user':u,'item':i,'utext':uh_t,'uimage':uh_i}

F_TRAIN=build_features(HIST_TRAIN); F_FULL=build_features(HIST_FULL)
# 训练期拟合 scaler；验证测试只能 transform。
U_SC=StandardScaler().fit(F_TRAIN['user']); I_SC=StandardScaler().fit(F_TRAIN['item'])
for ff in (F_TRAIN,F_FULL):
    ff['user']=U_SC.transform(ff['user']).astype(np.float32)
    ff['item']=I_SC.transform(ff['item']).astype(np.float32)

# 交叉塔输入：5 个统计特征 + 4 个匹配特征。
def _cross_numpy_block(users,items,ff):
    users=np.asarray(users,dtype=np.int64); items=np.asarray(items,dtype=np.int64)
    a=ff['user'][users]; b=ff['item'][items]
    sim_t=np.sum(ff['utext'][users]*TEXT[items],axis=1)
    sim_i=np.sum(ff['uimage'][users]*IMAGE[items],axis=1)
    # 热度匹配：用户历史活跃程度与候选商品热度差（代理特征）
    pop_diff=np.abs(a[:,0]-b[:,0]); rating_diff=np.abs(a[:,1]-b[:,1])
    return np.column_stack((a,b,sim_t,sim_i,pop_diff,rating_diff)).astype(np.float32)

def cross_numpy(users, items, ff):
    # Keep the original nine features; bound temporary high-dimensional arrays.
    result = np.empty((len(users), 9), dtype=np.float32)
    for start in range(0, len(users), 2048):
        end = min(start + 2048, len(users))
        result[start:end] = _cross_numpy_block(users[start:end], items[start:end], ff)
    return result

# 负例从用户在原始 TRAIN 已交互商品之外抽取，防止把已知正例标为负例。
seen_all={int(u):set(g.itemID.astype(int)) for u,g in TRAIN.groupby('userID')}
neg_per_pos=4
pairs=[]; labels=[]
for u,j in train_pos:
    pairs.append((u,j)); labels.append(1.)
    sampled=0
    while sampled<neg_per_pos:
        neg=int(np.random.randint(N_ITEMS))
        if neg not in seen_all[int(u)]:
            pairs.append((u,neg)); labels.append(0.); sampled+=1
pairs=np.asarray(pairs,np.int64); labels=np.asarray(labels,np.float32)
perm=np.random.permutation(len(pairs)); pairs=pairs[perm]; labels=labels[perm]

class PairDataset(Dataset):
    def __init__(self,pairs,labels,ff):
        self.pairs=pairs; self.labels=labels; self.cross=cross_numpy(pairs[:,0],pairs[:,1],ff)
    def __len__(self): return len(self.labels)
    def __getitem__(self,k):
        return (self.pairs[k,0],self.pairs[k,1],self.cross[k],self.labels[k])
loader=DataLoader(PairDataset(pairs,labels,F_TRAIN),batch_size=512,shuffle=True,num_workers=0)

class CrossNet(nn.Module):
    def __init__(self,d,layers):
        super().__init__(); self.ws=nn.ModuleList([nn.Linear(d,d) for _ in range(layers)])
    def forward(self,x):
        x0=x; h=x
        for layer in self.ws: h=h+x0*layer(h)
        return h

class ThreeTower(nn.Module):
    def __init__(self,kind):
        super().__init__(); self.kind=kind
        self.uid=nn.Embedding(N_USERS,32); self.iid=nn.Embedding(N_ITEMS,32)
        self.user_tower=nn.Sequential(nn.Linear(34,64),nn.ReLU(),nn.Linear(64,64))
        self.item_tower=nn.Sequential(nn.Linear(32+384+4096+3,128),nn.ReLU(),nn.Linear(128,64))
        d=9
        if kind=='none': self.cross_tower=None; cross_d=0
        elif kind=='linear': self.cross_tower=nn.Linear(d,32); cross_d=32
        elif kind=='mlp': self.cross_tower=nn.Sequential(nn.Linear(d,32),nn.ReLU(),nn.Linear(32,32)); cross_d=32
        elif kind=='fm':
            self.fm_v=nn.Parameter(torch.randn(d,16)*0.01)
            self.fm_linear=nn.Linear(d,1); self.fm_proj=nn.Linear(1,32); cross_d=32
        elif kind.startswith('dcn'):
            self.cross_tower=nn.Sequential(CrossNet(d,int(kind[-1])),nn.Linear(d,32)); cross_d=32
        self.head=nn.Sequential(nn.Linear(128+cross_d,64),nn.ReLU(),nn.Linear(64,1))
    def user_vec(self,u,uf): return self.user_tower(torch.cat([self.uid(u),uf],dim=1))
    def item_vec(self,i,stat,txt,img):
        return self.item_tower(torch.cat([self.iid(i),stat,txt,img],dim=1))
    def cross_vec(self,x):
        if self.kind=='none': return None
        if self.kind=='fm':
            z=x@self.fm_v
            interaction=.5*((z*z)-(x*x)@(self.fm_v*self.fm_v)).sum(dim=1,keepdim=True)
            return self.fm_proj(self.fm_linear(x)+interaction)
        return self.cross_tower(x)
    def forward_vectors(self,uv,iv,c):
        z=self.cross_vec(c)
        return self.head(torch.cat([uv,iv] if z is None else [uv,iv,z],dim=1)).squeeze(1)
    def forward(self,u,i,c,ff):
        uv=self.user_vec(u,ff['user'][u]); iv=self.item_vec(i,ff['item'][i],ff['text'][i],ff['image'][i])
        return self.forward_vectors(uv,iv,c)

# 特征常量放 GPU，训练时各实验复用。
def tensor_features(ff):
    return {'user':torch.from_numpy(ff['user']).to(DEVICE),
            'item':torch.from_numpy(ff['item']).to(DEVICE),
            'text':torch.from_numpy(TEXT).to(DEVICE),
            'image':torch.from_numpy(IMAGE).to(DEVICE)}
FT=tensor_features(F_TRAIN)
KINDS=['none','linear','mlp','fm','dcn2','dcn3']
MODELS={}; TRAIN_INFO={}
for kind in KINDS:
    torch.manual_seed(SEED)
    model=ThreeTower(kind).to(DEVICE)
    opt=torch.optim.Adam(model.parameters(),lr=1e-3,weight_decay=1e-5)
    loss_fn=nn.BCEWithLogitsLoss()
    start=time.perf_counter(); epoch_losses=[]
    for epoch in range(12):
        model.train(); total=0.
        for u,i,c,y in loader:
            u=u.to(DEVICE).long(); i=i.to(DEVICE).long(); c=c.to(DEVICE).float(); y=y.to(DEVICE).float()
            opt.zero_grad(set_to_none=True)
            loss=loss_fn(model(u,i,c,FT),y)
            loss.backward(); opt.step()
            total+=loss.item()*len(y)
        epoch_losses.append(total/len(pairs))
    elapsed=time.perf_counter()-start
    model.eval()
    checkpoint_path = RESULTS / f'threetower-{kind}.pt'
    torch.save(model.state_dict(), checkpoint_path)
    MODELS[kind] = checkpoint_path
    TRAIN_INFO[kind]={'OfflineTime(s)':elapsed,'Parameters':sum(p.numel() for p in model.parameters()),'TrainLoss':epoch_losses[-1]}
    print(f'{kind:7s} loss={epoch_losses[-1]:.4f} train={elapsed:.1f}s params={TRAIN_INFO[kind]["Parameters"]:,}')

    del model, opt, loss
    gc.collect()
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()
# Release training-only tensors before candidate evaluation.
del FT, loader
gc.collect()
if DEVICE.type == 'cuda': torch.cuda.empty_cache()


In [ ]:
# Cell 3 — Online + Evaluation：合并召回、缓存双塔向量、逐候选计算 Cross Tower + Head。
import glob
from collections import defaultdict

recall_files=sorted(RECALL_ROOT.glob('*.parquet'))
assert recall_files, f'未找到召回 Parquet：{RECALL_ROOT}'
required={'userID','itemID'}
frames=[]
for f in recall_files:
    t=pd.read_parquet(f, columns=['userID', 'itemID'])
    if required.issubset(t.columns):
        frames.append(t[['userID','itemID']].astype(np.int64))
assert frames,'召回文件必须包含 userID 和 itemID'
recall_frame_count = len(frames)
candidates=pd.concat(frames,ignore_index=True).drop_duplicates(['userID','itemID'])
del frames, t
gc.collect()
candidates=candidates[(candidates.userID>=0)&(candidates.userID<N_USERS)&(candidates.itemID>=0)&(candidates.itemID<N_ITEMS)]
# 过滤 TRAIN 已见商品；召回 Top-100 的并集可能大于 100。
seen_pairs=TRAIN[['userID','itemID']].drop_duplicates()
candidates=candidates.merge(seen_pairs.assign(_seen=1),on=['userID','itemID'],how='left')
candidates=candidates[candidates._seen.isna()][['userID','itemID']].reset_index(drop=True)
assert len(candidates)>0,'过滤历史交互后没有候选，请检查召回结果'
print('Recall files:',recall_frame_count,'candidate pairs:',len(candidates),'users:',candidates.userID.nunique())

# 为避免 4096D 图片特征重复推理，先缓存全部商品塔输出；用户塔每个用户算一次。
FF=tensor_features(F_FULL)
users=candidates.userID.to_numpy(dtype=np.int64); items=candidates.itemID.to_numpy(dtype=np.int64)
CROSS=cross_numpy(users,items,F_FULL)

def score_candidates(model,batch=2048):
    model.eval(); start=time.perf_counter()
    with torch.no_grad():
        all_items=[]
        for s in range(0,N_ITEMS,256):
            ix=torch.arange(s,min(s+256,N_ITEMS),device=DEVICE)
            all_items.append(model.item_vec(ix,FF['item'][ix],FF['text'][ix],FF['image'][ix]).cpu())
        item_cache=torch.cat(all_items)
        active=np.unique(users)
        user_cache=torch.zeros((N_USERS,64),dtype=torch.float32)
        for chunk in np.array_split(active,max(1,math.ceil(len(active)/2048))):
            ix=torch.as_tensor(chunk,device=DEVICE)
            user_cache[chunk]=model.user_vec(ix,FF['user'][ix]).cpu()
        cache_time=time.perf_counter()-start
        start_online=time.perf_counter(); scores=[]
        for s in range(0,len(users),batch):
            e=min(s+batch,len(users)); u=users[s:e]; j=items[s:e]
            uv=user_cache[u].to(DEVICE); iv=item_cache[j].to(DEVICE)
            cv=torch.from_numpy(CROSS[s:e]).to(DEVICE)
            scores.append(model.forward_vectors(uv,iv,cv).cpu().numpy())
        if DEVICE.type=='cuda': torch.cuda.synchronize()
        online_time=time.perf_counter()-start_online
    return np.concatenate(scores),cache_time,online_time

EVAL_K=[5,10,20,50,100]
truth={
 'Validation':{int(u):set(g.itemID.astype(int)) for u,g in VAL.groupby('userID')},
 'Test':{int(u):set(g.itemID.astype(int)) for u,g in TEST.groupby('userID')}
}
train_pop=TRAIN.groupby('itemID').size().to_dict()
train_catalog=set(train_pop)

def evaluate_ranked(ranked,ground,ks=EVAL_K):
    # 与召回实验一致：对 ground 中全部用户平均，未被召回的用户按零分计。
    rows=[]; n=len(ground)
    for k in ks:
        sums=defaultdict(float); exposed=set()
        for u,positives in ground.items():
            rec=ranked.get(u,[])[:k]; hits=np.array([int(j in positives) for j in rec],dtype=float)
            n_hit=float(hits.sum()); exposed.update(rec)
            sums['Recall']+=n_hit/max(len(positives),1)
            sums['Precision']+=n_hit/k
            sums['HitRate']+=float(n_hit>0)
            dcg=float(np.sum(hits/np.log2(np.arange(2,len(hits)+2))))
            ideal=float(np.sum(1/np.log2(np.arange(2,min(k,len(positives))+2))))
            sums['NDCG']+=dcg/ideal if ideal else 0.
            sums['MAP']+=float(np.sum((np.cumsum(hits)/np.arange(1,len(hits)+1))*hits))/max(min(k,len(positives)),1)
            hitidx=np.flatnonzero(hits)
            sums['MRR']+=1/(int(hitidx[0])+1) if len(hitidx) else 0.
            sums['Novelty']+=np.mean([-np.log2((train_pop.get(j,0)+1)/(N_USERS+1)) for j in rec]) if rec else 0.
        row={'K':k,**{name:sums[name]/max(n,1) for name in ('Recall','NDCG','Precision','HitRate','MAP','MRR','Novelty')}}
        row['Coverage']=len(exposed)/max(len(train_catalog),1)
        rows.append(row)
    return rows

all_metrics=[]; unique_users=max(candidates.userID.nunique(),1)
for kind in KINDS:
    model=ThreeTower(kind).to(DEVICE)
    model.load_state_dict(torch.load(MODELS[kind], map_location=DEVICE, weights_only=True))
    logits,cache_s,online_s=score_candidates(model)
    scored=candidates.copy(); scored['score']=logits.astype(float)
    scored=scored.sort_values(['userID','score','itemID'],ascending=[True,False,True])
    scored['rank']=scored.groupby('userID').cumcount()+1
    scored=scored[scored['rank']<=100].copy()
    channel='threetower-'+kind; scored['channel']=channel
    scored[['userID','itemID','score','rank','channel']].to_parquet(OUT/f'{channel}.parquet',index=False)
    ranked={int(u):g.itemID.astype(int).tolist() for u,g in scored.groupby('userID',sort=False)}
    for split,gt in truth.items():
        rows=evaluate_ranked(ranked,gt)
        candidate_gt={int(u):set(g.itemID.astype(int)) for u,g in candidates.groupby('userID')}
        ceiling=np.mean([len(positive & candidate_gt.get(u,set()))/max(len(positive),1) for u,positive in gt.items()])
        for row in rows:
            row.update({'Experiment':channel,'Split':split,'CandidateRecallCeiling':ceiling,
                        'OfflineTime(s)':TRAIN_INFO[kind]['OfflineTime(s)'],
                        'Parameters':TRAIN_INFO[kind]['Parameters'],
                        'CacheBuildTime(s)':cache_s,'OnlineTime(s)':online_s,
                        'Latency(ms/user)':online_s/unique_users*1000,
                        'Users/s':unique_users/max(online_s,1e-9),
                        'AvgCandidates':len(candidates)/unique_users,
                        'UsersWith100':int((scored.groupby('userID').size()>=100).sum()),
                        'EmptyUsers':int(len(gt)-len(set(gt)&set(ranked)))})
            all_metrics.append(row)
    print(channel,'cached=',round(cache_s,2),'online=',round(online_s,2),'s')

    del model, scored, logits, ranked
    gc.collect()
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()

metrics=pd.DataFrame(all_metrics)
metrics.to_csv(RESULTS/'threetower_all_metrics.csv',index=False)
summary=metrics[(metrics.Split=='Validation')&(metrics.K==20)].sort_values('NDCG',ascending=False)
print('\nValidation @20 (selection only):')
display(summary[['Experiment','Recall','NDCG','HitRate','CandidateRecallCeiling','Latency(ms/user)','Parameters']].round(4))
best=summary.iloc[0]['Experiment']
print('\nBest on Validation:',best)
print('Test @20 (report only):')
display(metrics[(metrics.Split=='Test')&(metrics.K==20)&(metrics.Experiment==best)][['Experiment','Recall','NDCG','HitRate','CandidateRecallCeiling']].round(4))
print('Saved:',OUT,'and',RESULTS/'threetower_all_metrics.csv')


K,Recall,NDCG,Precision,HitRate,MAP,MRR,Novelty,Coverage,Experiment,Split,CandidateRecallCeiling,OfflineTime(s),Parameters,CacheBuildTime(s),OnlineTime(s),Latency(ms/user),Users/s,AvgCandidates,UsersWith100,EmptyUsers
5,0.008406,0.004769,0.001790,0.008948,0.003518,0.003701,7.326332,0.184192,threetower-none,Validation,0.498811,12.352582,1448865,0.281309,8.144119,0.418828,2387.612453,1054.681306,19445,0
10,0.018366,0.008001,0.001944,0.019388,0.004828,0.005075,7.347507,0.297148,threetower-none,Validation,0.498811,12.352582,1448865,0.281309,8.144119,0.418828,2387.612453,1054.681306,19445,0
20,0.034758,0.012119,0.001833,0.036513,0.005913,0.006204,7.651514,0.466582,threetower-none,Validation,0.498811,12.352582,1448865,0.281309,8.144119,0.418828,2387.612453,1054.681306,19445,0
50,0.070421,0.019246,0.001491,0.074055,0.007029,0.007378,8.237298,0.758479,threetower-none,Validation,0.498811,12.352582,1448865,0.281309,8.144119,0.418828,2387.612453,1054.681306,19445,0
100,0.113517,0.026282,0.001205,0.118591,0.007634,0.007996,8.689509,0.940116,threetower-none,Validation,0.498811,12.352582,1448865,0.281309,8.144119,0.418828,2387.612453,1054.681306,19445,0
5,0.010149,0.005702,0.002181,0.010903,0.004141,0.004476,7.326332,0.184192,threetower-none,Test,0.489317,12.352582,1448865,0.281309,8.144119,0.418828,2387.612453,1054.681306,19445,0
10,0.020936,0.009233,0.002283,0.022782,0.005551,0.006024,7.347507,0.297148,threetower-none,Test,0.489317,12.352582,1448865,0.281309,8.144119,0.418828,2387.612453,1054.681306,19445,0
20,0.037808,0.013573,0.002073,0.041245,0.006712,0.007287,7.651514,0.466582,threetower-none,Test,0.489317,12.352582,1448865,0.281309,8.144119,0.418828,2387.612453,1054.681306,19445,0
50,0.072106,0.020494,0.001591,0.078992,0.007785,0.008466,8.237298,0.758479,threetower-none,Test,0.489317,12.352582,1448865,0.281309,8.144119,0.418828,2387.612453,1054.681306,19445,0
100,0.115962,0.027761,0.001292,0.126614,0.008407,0.009131,8.689509,0.940116,threetower-none,Test,0.489317,12.352582,1448865,0.281309,8.144119,0.418828,2387.612453,1054.681306,19445,0


In [ ]:
# Verify every requested output before marking the run complete.
import hashlib
import json
expected_recall = (
    [f'itemcf-{mode}-nb{nb}.parquet' for mode in ['binary','rating'] for nb in [10,50,100,'all']]
    + [f'swing-{mode}-a1-nb{nb}.parquet' for mode in ['binary','rating'] for nb in [10,50,100]]
    + [f'usercf-{method}-nb{nb}.parquet' for method in ['cosine','pop-penalty'] for nb in [10,50,100]]
    + [cfg['name'] + '.parquet' for cfg in EXPERIMENTS]
)
expected_prerank = [f'threetower-{kind}.parquet' for kind in KINDS]
for directory, names in [(RECALL_ROOT, expected_recall), (OUT, expected_prerank)]:
    assert all((directory / name).is_file() for name in names), (directory, names)
    for name in names:
        frame = pd.read_parquet(directory / name)
        assert {'userID','itemID','score','rank','channel'}.issubset(frame.columns)
        assert len(frame) > 0 and frame['rank'].between(1,100).all(), name
files = []
for path in sorted(WORK.rglob('*')):
    if path.is_file():
        digest = hashlib.sha256()
        with path.open('rb') as stream:
            for block in iter(lambda: stream.read(1024*1024), b''):
                digest.update(block)
        files.append({'path': str(path.relative_to(WORK)), 'bytes': path.stat().st_size,
                      'sha256': digest.hexdigest()})
(WORK / 'run_summary.json').write_text(json.dumps({
    'status': 'complete', 'recall_experiments': len(expected_recall),
    'prerank_experiments': len(expected_prerank), 'files': files
}, indent=2))
print('COMPLETE:', len(expected_recall), 'recall experiments;', len(expected_prerank), 'pre-ranking experiments')
